# Phase 3 — 04b: teacher trace generation (DeepSeek, CPU)

**Runtime: CPU** (no GPU — this is API calls + test-running). Safe to run **in
parallel** with notebook 04: separate VM, near-zero Colab units. Cost is DeepSeek
API money instead — expect well under $1 for all 544 train bugs.

**What it does (Amendment A2, trace arm):** for every TRAIN-split bug, the
DeepSeek teacher writes a **short 2–4 sentence diagnosis + a fixed function**.
We keep a trace only if it is *verified*: the fix must pass the **full** test
suite by execution, and the diagnosis must stay short. Up to 3 tries per bug.

**Ground rules baked in:**
- The teacher **never sees the gold fix** — same repair prompt the student gets
  (3 visible tests), graded on the full suite afterward.
- API key comes from **Colab Secrets only** (`DEEPSEEK_API_KEY`) — never typed
  into a cell. This notebook gets committed to a public repo.
- Nothing HumanEval-derived is touched; train bugs only.

In [ ]:
# Setup: Drive, fresh repo, canonical prompt module, the bug corpus
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, importlib
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
os.makedirs(PHASE3, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
train_bugs = [b for b in bugs if b['split'] == 'train']
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
print(len(train_bugs), 'train bugs to trace |', len(routing), 'routed')

In [ ]:
# DeepSeek client — key from Colab Secrets ONLY (key icon in the left sidebar:
# add secret DEEPSEEK_API_KEY, paste the key there, toggle 'Notebook access' ON)
%pip install -q openai
from google.colab import userdata
from openai import OpenAI
client = OpenAI(api_key=userdata.get('DEEPSEEK_API_KEY'),
                base_url='https://api.deepseek.com')
ping = client.chat.completions.create(model='deepseek-chat', max_tokens=5,
    messages=[{'role': 'user', 'content': 'Reply with exactly: OK'}])
print('Auth OK — teacher says:', ping.choices[0].message.content.strip())

In [ ]:
# Teacher call + execution verification (teacher NEVER sees the gold fix)
import subprocess, tempfile, time

TEACHER_SUFFIX = """

First write a 2-4 sentence diagnosis: what is wrong in the code and why it
fails the tests. Then give the complete fixed function in a single ```python
block. Nothing after the code block."""

def ask_teacher(bug, temperature):
    prompt = build_repair_prompt(bug['buggy'], bug['test_list']) + TEACHER_SUFFIX
    r = client.chat.completions.create(model='deepseek-chat', max_tokens=1024,
        temperature=temperature, messages=[{'role': 'user', 'content': prompt}])
    text = r.choices[0].message.content
    diagnosis = text.split('```')[0].strip()
    return diagnosis, extract_code(text), r.usage.prompt_tokens, r.usage.completion_tokens

def passes(code, bug, timeout=6):
    prog = '\n'.join(list(bug['test_imports']) + [code] + list(bug['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

def trace_for_bug(bug, temps=(0.0, 0.8, 1.0), max_words=120):
    tin = tout = 0
    for n, t in enumerate(temps, 1):
        try:
            diag, code, i, o = ask_teacher(bug, t)
        except Exception:
            time.sleep(3); continue
        tin += i; tout += o
        if diag and len(diag.split()) <= max_words and passes(code, bug):
            return {'id': bug['id'], 'ok': True, 'diagnosis': diag, 'fix': code,
                    'attempts': n, 'in_tokens': tin, 'out_tokens': tout}
    return {'id': bug['id'], 'ok': False, 'attempts': len(temps),
            'in_tokens': tin, 'out_tokens': tout}

In [ ]:
# SMOKE: 2 bugs — eyeball the diagnosis quality before the full run
for b in train_bugs[:2]:
    r = trace_for_bug(b)
    print('=' * 60)
    print(b['id'], '|', b['description'], '| verified:', r['ok'], '| attempts:', r['attempts'])
    if r['ok']:
        print('DIAGNOSIS:', r['diagnosis'])
print('\nHealthy = a crisp 2-4 sentence diagnosis naming the ACTUAL bug.')
print('If it rambles, is generic, or verification keeps failing: STOP and report.')

In [ ]:
# FULL RUN with checkpoint-resume (saves every 20 bugs; safe to re-run anytime)
from concurrent.futures import ThreadPoolExecutor
CKPT = f'{PHASE3}/traces_ckpt.json'
done = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
todo = [b for b in train_bugs if b['id'] not in done]
print(f'resuming: {len(done)} done, {len(todo)} to go')
t0 = time.time()
for i in range(0, len(todo), 20):
    batch = todo[i:i+20]
    with ThreadPoolExecutor(max_workers=4) as pool:
        for r in pool.map(trace_for_bug, batch):
            done[r['id']] = r
    json.dump(done, open(CKPT, 'w'))
    ok = sum(1 for r in done.values() if r['ok'])
    print(f'{len(done)}/{len(train_bugs)} traced | {ok} verified | {time.time()-t0:.0f}s elapsed')
print('All attempts checkpointed to', CKPT)

In [ ]:
# THE REPORT: coverage, cost, and the final trace file for notebook 04c
ok = [r for r in done.values() if r['ok']]
total = len(done)
print(f'=== COVERAGE: {len(ok)}/{total} train bugs have a verified short trace ({len(ok)/total*100:.1f}%) ===')
print('\n=== by routing pile (sft pile matters most — those NEED worked solutions) ===')
for pile in ('sft', 'rl', 'easy'):
    ids = [i for i in done if routing.get(i) == pile]
    got = sum(1 for i in ids if done[i]['ok'])
    print(f"  {pile:<5} {got:>3}/{len(ids):<3}  ({got/max(len(ids),1)*100:.0f}%)")
by_id = {b['id']: b for b in train_bugs}
print('\n=== by category ===')
for cat in sorted({b['category'] for b in train_bugs}):
    ids = [i for i in done if by_id[i]['category'] == cat]
    got = sum(1 for i in ids if done[i]['ok'])
    print(f"  {cat:<18} {got:>3}/{len(ids)}")
words = [len(r['diagnosis'].split()) for r in ok]
tin = sum(r['in_tokens'] for r in done.values())
tout = sum(r['out_tokens'] for r in done.values())
print(f'\nmean diagnosis length: {sum(words)/len(words):.0f} words (cap 120)')
print(f'API usage: {tin:,} in + {tout:,} out tokens  ~ ${tin/1e6*0.27 + tout/1e6*1.10:.2f}')
json.dump(ok, open(f'{PHASE3}/traces_v0.json', 'w'), indent=1)
print('\nFinal verified traces saved to', f'{PHASE3}/traces_v0.json')

## Bring back to the session
1. The **coverage line** (how many of 544 got a verified trace)
2. The **sft-pile coverage** — those 97 bugs are the ones the student can't solve;
   a teacher trace is the only worked solution they'll ever get
3. The **smoke-cell diagnoses** if anything looked unhealthy
4. The **cost line**

Bugs with no verified trace simply fall back to plain gold-fix targets in the
trace-arm mixture (notebook 04c decides that). After this: 04c — trace-arm SFT,
then the A/B verdict against notebook 04's no-trace run.